In [1]:
# B.11  Finding choices of cyclotomic and fields

# [LS18, eprint 2017-523] pg 6
# m is the cyclotomic polynomial index
def tau(m):
    return m if (m % 2) != 0 else m / 2


# [LS18, eprint 2017-523] Thm 1.1, pg 4
# m is the cyclotomic polynomial index
# p is the prime
# z is any divisor of m
# This tests for the condition for thm 1.1 to hold
def thm1_1_cond(m, p, z):
    cond1 = (p % z) == 1
    cond2 = Mod(p, m).multiplicative_order() == m / z
    return cond1 and cond2


# [LS18, eprint 2017-523] Thm 1.1, pg 4
# p is the prime
# z is any divisor of m
# lInf bound for elements to be invertible
# given that m,p,z satisfy thm 1.1 cond
def thm1_1_inv_bound(p, z):
    return (1 / s1(z) * p^(1 / euler_phi(z))).n()


def thm1_1_num_factors(z):
    return euler_phi(z)


# Output divisors of m
def divisors(m):
    zs = list()
    for i in range(1, m + 1):
        if m % i == 0:
            zs.append(i)
    return zs


# [LS18, eprint 2017-523] pg 6, pg 9
# We only consider prime power cyclotomics
# m is the cyclotomic polynomial index
def s1(m):
    return sqrt(tau(m))


# checks if cyclotomic index m is power of two
def is_pow2(m):
    return sum(m.digits(2)) == 1


# [MR09] lattice-based cryptography

# makes sure characteristic does not lead
# to trivial bound
def non_trivial(q, n, d, delta):
    return (q / 2).n() >= (2^(2 * sqrt(n * d * log(q, 2) * log(delta, 2)))).n()


# [AL21] eprint Prop 2. 2021/202
# for all u,v in R, |u*v| / |v| <= gamma*|u|
# outputs T = gamma * |u|
# assumes we are only testing prime powers
def expansion_factor(m, norm):
    if is_pow2(m):
        return euler_phi(m) * norm
    else:
        return 2 * euler_phi(m) * norm


# p is prime
# max_idx is max cyclotomic index
# outputs list of (m, z)
def candidates(p, min_idx=10, max_idx=200):
    # prime powers
    possible_indices = [i for i in range(min_idx, max_idx) if len(factor(i)) == 1]
    c = list()
    for m in possible_indices:
        zs = divisors(m)
        for z in zs:
            if thm1_1_cond(m, p, z):
                c.append((Integer(m), Integer(z)))
    return c


def pre_filter(q, cyclotomic_index, z, n, m, chals):
    chals_norm = max([abs(c) for c in chals])
    chals_max_diff = chals[-1] - chals[0]
    delta = 1.0045  # root hermite factor, chosen from [ESSL19] eprint 2018/773
    phi = cyclotomic_polynomial(cyclotomic_index)  # index cyclotomic polynomial
    d = phi.degree()  # degree of cyclotomic
    # return non_trivial(q, n, d, delta) and chals_max_diff < thm1_1_inv_bound(q, z) and log(len(chals)^d,2).n() >= 120
    # We remove non_trivial(...) because we use the lattice estimator for hardness
    return chals_max_diff < thm1_1_inv_bound(q, z) and log(len(chals)^d, 2).n() >= 120


def info(q, cyclotomic_index, z, n, m, chals):
    chals_norm = max([abs(c) for c in chals])
    chals_max_diff = chals[-1] - chals[0]
    delta = 1.0045  # root hermite factor, chosen from [ESSL19] eprint 2018/773
    phi = cyclotomic_polynomial(cyclotomic_index)  # index cyclotomic polynomial
    d = phi.degree()  # degree of cyclotomic
    T = expansion_factor(cyclotomic_index, chals_norm)

    # Bounds for MSIS to be hard
    # [MR09] lattice-based cryptography pg 6

    # [CMMW24] pg 38 eprint 2024/281
    MSIS_B_L2_bound = min(q, 2^(2 + sqrt(n * d * log(q, 2) * log(delta, 2))))
    MSIS_B_inf_bound = MSIS_B_L2_bound / sqrt(m * d)

    # We need MSIS infinity bound 8T B to be hard
    B = MSIS_B_inf_bound / (8 * T)

    print("####")
    print("Cyclotomic idx:", cyclotomic_index)
    print("Cyclotomic Poly:", phi)
    print("z:", z)
    # print("Prime is non-trivial?", non_trivial(q, n, d, delta))
    print("Small norm is small enough?", chals_max_diff < thm1_1_inv_bound(q, z))
    print("Csmall large enough?", log(len(chals)^d, 2).n() >= 120)
    print("Degree of Cyclotomic:", d)
    # print("log(B):", log(B,2).n())
    print("Expansion Factor T:", T)
    print("Invertible Norm bound:", thm1_1_inv_bound(q, z))
    print("log(|C_small|^d):", log(len(chals)^d, 2).n())
    print("Factors of Cyclotomic:", thm1_1_num_factors(z))
    print()


def possible_settings(q, n, m, chals):
    for (cyclotomic_index, z) in candidates(q):
        if pre_filter(q, cyclotomic_index, z, n, m, chals):
            info(q, cyclotomic_index, z, n, m, chals)
        else:
            delta = 1.0045
            d = cyclotomic_polynomial(cyclotomic_index).degree()
            print("[Does not satisfy security requirements] index: {}, degree: {}, z: {}, non_trivial: {}, Csmall large enough: {}, chals_max_diff: {}, chals_norm: {}".format(
                cyclotomic_index,
                d,
                z,
                non_trivial(q, n, d, delta),
                log(len(chals)^d, 2).n() >= 120,
                chals[-1] - chals[0],
                max([abs(c) for c in chals]),
            ))
            print()


# Primes:
GL = 2^64 - 2^32 + 1
AGL = GL - 32

print("###########################")
print("AGL ###########################")
print("###########################")
# MSIS settings
n = 13  # # rows, kappa in latticefold
m = 2^26  # # cols
# Small Challenge set
chals = [-1, 0, 1, 2]
possible_settings(AGL, n, m, chals)

print("###########################")
print("M61 ###########################")
print("###########################")
# MSIS settings
n = 16  # # rows, kappa in latticefold
m = 2^22  # # cols
# Small Challenge set
chals = [-2, -1, 0, 1, 2]
possible_settings(2^61 - 1, n, m, chals)

print("###########################")
print("GL ###########################")
print("###########################")
# MSIS settings
n = 16  # # rows, kappa in latticefold
m = 2^24  # # cols
# Small Challenge set
chals = [-2, -1, 0, 1, 2]
possible_settings(GL, n, m, chals)
print("###########################")


###########################
AGL ###########################
###########################
[Does not satisfy security requirements] index: 16, degree: 8, z: 16, non_trivial: True, Csmall large enough: False, chals_max_diff: 3, chals_norm: 2

[Does not satisfy security requirements] index: 32, degree: 16, z: 32, non_trivial: True, Csmall large enough: False, chals_max_diff: 3, chals_norm: 2

[Does not satisfy security requirements] index: 64, degree: 32, z: 32, non_trivial: True, Csmall large enough: False, chals_max_diff: 3, chals_norm: 2

####
Cyclotomic idx: 128
Cyclotomic Poly: x^64 + 1
z: 32
Small norm is small enough? True
Csmall large enough? True
Degree of Cyclotomic: 64
Expansion Factor T: 128
Invertible Norm bound: 3.99999999994179
log(|C_small|^d): 128.000000000000
Factors of Cyclotomic: 16

###########################
M61 ###########################
###########################
[Does not satisfy security requirements] index: 11, degree: 10, z: 11, non_trivial: True, Csmall large

In [2]:
BB = 2^31 - 2^27 + 1

print("###########################")
print("BB ###########################")
print("###########################")
# MSIS settings
n = 16  # # rows, kappa in latticefold
m = 2^20  # # cols
# Small Challenge set
chals = [-2, -1, 0, 1, 2]
possible_settings(BB, n, m, chals)

###########################
BB ###########################
###########################
[Does not satisfy security requirements] index: 16, degree: 8, z: 16, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 25, degree: 20, z: 5, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 27, degree: 18, z: 3, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 32, degree: 16, z: 32, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 64, degree: 32, z: 64, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

####
Cyclotomic idx: 81
Cyclotomic Poly: x^54 + x^27 + 1
z: 3
Small norm is small enough? True
Csmall large enough? True
Degree of Cyclotomic: 54
Expansion 

In [6]:
# B.12  Lattice Estimator Script

import sys
sys.path.append("/Users/clankpan/Develop/ZKP/lattice-estimator")

from estimator import *
Logging.set_level(Logging.LEVEL0)

M61 = 2^61 - 1
GL  = 2^64 - 2^32 + 1
AGL = GL - 32

n = 13
d = 64
T = 128
k = 11
b = 2
B = b^k
m = 2^26
q = AGL

n_sis = n*d
m_sis = m*d
B_L2  = sqrt(m*d) * (8*T*B)

params = SIS.Parameters(n=n_sis, q=q, m=m_sis, length_bound=B_L2, norm=2)
_ = SIS.estimate(params)
print((k+1)*T*(b-1) < B)


n = 16
d = 54
T = 216
k = 12
b = 2
B = b^k
m = 2^22
q = M61

n_sis = n*d
m_sis = m*d
B_L2  = sqrt(m*d) * (8*T*B)

params = SIS.Parameters(n=n_sis, q=q, m=m_sis, length_bound=B_L2, norm=2)
_ = SIS.estimate(params)
print((k+1)*T*(b-1) < B)


n = 16
d = 54
T = 216
k = 12
b = 2
B = b^k
m = 2^24
q = GL

n_sis = n*d
m_sis = m*d
B_L2  = sqrt(m*d) * (8*T*B)

params = SIS.Parameters(n=n_sis, q=q, m=m_sis, length_bound=B_L2, norm=2)
_ = SIS.estimate(params)
print((k+1)*T*(b-1) < B)


lattice  :: rop: ≈2^127.3, red: ≈2^127.3, δ: 1.004461, β: 339, d: 2878, tag: euclidean
True
lattice  :: rop: ≈2^128.7, red: ≈2^128.7, δ: 1.004417, β: 344, d: 2877, tag: euclidean
True
lattice  :: rop: ≈2^127.9, red: ≈2^127.9, δ: 1.004443, β: 341, d: 2938, tag: euclidean
True


In [7]:
# BabyBear の設定を試す
BB = 2^31 - 2^27 + 1

n = 16
d = 54       # idx:81 の Degree of Cyclotomic
T = 216      # idx:81 の Expansion Factor T
b = 2
m = 2^20     # とりあえずの列数、後で増減して良い

for k in range(8, 15):   # B = 2^k を色々スキャン
    B = b^k
    n_sis = n*d
    m_sis = m*d
    B_L2  = sqrt(m*d) * (8*T*B)  # （論文の仮定ならこれでOK）

    print("### BabyBear, k =", k)
    params = SIS.Parameters(n=n_sis, q=BB, m=m_sis, length_bound=B_L2, norm=2)
    _ = SIS.estimate(params)
    print((k+1)*T*(b-1) < B)
    print()


### BabyBear, k = 8
Algorithm functools.partial(<estimator.sis_lattice.SISLattice object at 0x13b62b380>, red_cost_model=<estimator.reduction.MATZOV object at 0x13b628980>, red_shape_model='gsa') on SISParameters(n=864, q=2013265921, length_bound=1358954496*sqrt(6), m=56623104, norm=2, tag=None) failed with SIS trivially easy. Please set norm bound < (q-1)/2.
False

### BabyBear, k = 9
Algorithm functools.partial(<estimator.sis_lattice.SISLattice object at 0x13b62b380>, red_cost_model=<estimator.reduction.MATZOV object at 0x13b628980>, red_shape_model='gsa') on SISParameters(n=864, q=2013265921, length_bound=2717908992*sqrt(6), m=56623104, norm=2, tag=None) failed with SIS trivially easy. Please set norm bound < (q-1)/2.
False

### BabyBear, k = 10
Algorithm functools.partial(<estimator.sis_lattice.SISLattice object at 0x13b62b380>, red_cost_model=<estimator.reduction.MATZOV object at 0x13b628980>, red_shape_model='gsa') on SISParameters(n=864, q=2013265921, length_bound=5435817984*sqr

In [9]:
KB = 2^31 - 2^24 + 1

print("###########################")
print("KB ###########################")
print("###########################")
# MSIS settings
n = 16  # # rows, kappa in latticefold
m = 2^20  # # cols
# Small Challenge set
chals = [-2, -1, 0, 1, 2]
possible_settings(KB, n, m, chals)

###########################
KB ###########################
###########################
[Does not satisfy security requirements] index: 16, degree: 8, z: 16, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 32, degree: 16, z: 32, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 64, degree: 32, z: 64, non_trivial: True, Csmall large enough: False, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 127, degree: 126, z: 127, non_trivial: False, Csmall large enough: True, chals_max_diff: 4, chals_norm: 2

[Does not satisfy security requirements] index: 128, degree: 64, z: 128, non_trivial: True, Csmall large enough: True, chals_max_diff: 4, chals_norm: 2

